**Install Feast**

In [ ]:
!pip -q install feast==0.64.0 pyarrow scikit-learn pandas numpy


In [ ]:
import feast

print("Feast version:", feast.__version__)


**Upload the CSE Graduate Skill-Gap Dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
import pandas as pd

# Use the uploaded CSV file.
# If your uploaded filename is different, change the filename below.
df = pd.read_csv("curriculum_industry_skill_gap_500_samples(1).csv")

print(df.head())
print("Shape:", df.shape)


**Examine the Data**

In [ ]:
print(df.columns)
print(df.info())
print(df.isnull().sum())


**Create Features - perform feature engineering for the CSE graduate skill-gap dataset.**

In [ ]:
import pandas as pd

# -----------------------------
# ENTITY
# -----------------------------
df["skill_id"] = df["skill_id"].astype(str)

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
# Numeric feature types required by Feast
float_features = [
    "curriculum_score",
    "industry_score",
    "practical_score",
    "coding_score",
    "communication_score",
    "problem_solving_score",
    "teamwork_score",
    "industry_demand_score",
    "curriculum_relevance_score",
    "skill_gap",
    "employability_score"
]

int_features = [
    "projects_completed",
    "internships_completed",
    "certifications"
]

for col in float_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float32")

for col in int_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int64")

# Recalculate the skill gap from curriculum and industry scores
# so the feature engineering step is explicit.
df["skill_gap"] = (
    df["industry_score"] - df["curriculum_score"]
).clip(lower=0).astype("float32")

# Gap level derived from skill gap
def gap_level(gap):
    if gap >= 25:
        return "High"
    elif gap >= 12:
        return "Medium"
    return "Low"

df["gap_level"] = df["skill_gap"].apply(gap_level).astype(str)

# Target
# 1 = employability condition satisfied, 0 = otherwise
df["employability_target"] = df["employability_target"].astype("int64")

print("Feature engineering completed.")


In [ ]:
df[
    [
        "skill_id",
        "skill_name",
        "curriculum_score",
        "industry_score",
        "practical_score",
        "skill_gap",
        "gap_level",
        "employability_score",
        "employability_target"
    ]
].head()


**Add Event Timestamps** Feast models feature values with timestamps so historical retrieval can perform point-in-time joins. The skill-gap dataset is not naturally a time-series dataset, so for this classroom demonstration we create artificial timestamps.

In [ ]:
base_time = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time +
    pd.to_timedelta(
        df.index,
        unit="s"
    )
)

df["created_timestamp"] = (
    df["event_timestamp"] +
    pd.Timedelta(seconds=1)
)


**Separate Features and Labels**

In [ ]:
feature_df = df[
    [
        "skill_id",
        "event_timestamp",
        "created_timestamp",
        "curriculum_score",
        "industry_score",
        "practical_score",
        "projects_completed",
        "internships_completed",
        "certifications",
        "coding_score",
        "communication_score",
        "problem_solving_score",
        "teamwork_score",
        "industry_demand_score",
        "curriculum_relevance_score",
        "skill_gap",
        "employability_score",
        "gap_level"
    ]
].copy()


In [ ]:
label_df = df[
    [
        "skill_id",
        "event_timestamp",
        "employability_target"
    ]
].copy()


In [ ]:
print("Feature data:")
display(feature_df.head())

print("Labels:")
display(label_df.head())


**Create the Feast Repository**

In [ ]:
import os

repo_path = "/content/cse_skillgap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [ ]:
# Re-run-safe repository creation
import os

repo_path = "/content/cse_skillgap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [ ]:
feature_df.to_parquet(
    f"{repo_path}/data/cse_skillgap_features.parquet",
    index=False
)

print("Parquet feature file created.")


Create feature_store.yaml

In [ ]:
feature_store_yaml = """
project: cse_skillgap_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)

print("feature_store.yaml created.")


**Define the Entity and Feature View** Create: features.py

In [ ]:
feature_definition = '''
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64,
    String
)


# -----------------------------
# ENTITY
# -----------------------------

skill = Entity(
    name="skill",
    join_keys=["skill_id"],
    description="Unique CSE graduate skill-gap entity"
)


# -----------------------------
# DATA SOURCE
# -----------------------------

skillgap_source = FileSource(
    name="cse_skillgap_source",
    path="data/cse_skillgap_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)


# -----------------------------
# FEATURE VIEW
# -----------------------------

skillgap_feature_view = FeatureView(
    name="cse_skillgap_features",
    entities=[skill],

    ttl=timedelta(days=3650),

    schema=[
        Field(name="curriculum_score", dtype=Float32),
        Field(name="industry_score", dtype=Float32),
        Field(name="practical_score", dtype=Float32),
        Field(name="projects_completed", dtype=Int64),
        Field(name="internships_completed", dtype=Int64),
        Field(name="certifications", dtype=Int64),
        Field(name="coding_score", dtype=Float32),
        Field(name="communication_score", dtype=Float32),
        Field(name="problem_solving_score", dtype=Float32),
        Field(name="teamwork_score", dtype=Float32),
        Field(name="industry_demand_score", dtype=Float32),
        Field(name="curriculum_relevance_score", dtype=Float32),
        Field(name="skill_gap", dtype=Float32),
        Field(name="employability_score", dtype=Float32),
        Field(name="gap_level", dtype=String),
    ],

    source=skillgap_source,
    online=True
)


# -----------------------------
# FEATURE SERVICE
# -----------------------------

cse_skillgap_service = FeatureService(
    name="cse_skillgap_prediction_service",
    features=[skillgap_feature_view]
)
'''

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

print("features.py created.")


In [ ]:
!find /content/cse_skillgap_feast -maxdepth 2 -type f

**Apply the Feature Definitions**

In [ ]:
%cd /content/cse_skillgap_feast

In [ ]:
!feast apply

In [ ]:
!feast entities list

In [ ]:
!feast feature-views list

**Create the Feast Store Object**

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/cse_skillgap_feast"
)


In [ ]:
feature_service = store.get_feature_service(
    "cse_skillgap_prediction_service"
)


**Retrieve Historical Features**

In [ ]:
entity_df = label_df.copy()

display(entity_df.head())


In [ ]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()


In [ ]:
display(training_data.head())
print("Historical feature shape:", training_data.shape)


**Train a Model Using Feast Features**

In [ ]:
feature_columns = [
    "curriculum_score",
    "industry_score",
    "practical_score",
    "projects_completed",
    "internships_completed",
    "certifications",
    "coding_score",
    "communication_score",
    "problem_solving_score",
    "teamwork_score",
    "industry_demand_score",
    "curriculum_relevance_score",
    "skill_gap",
    "employability_score"
]


In [ ]:
X = training_data[feature_columns]

y = training_data["employability_target"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)


In [ ]:
predictions = model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)


The important point is not only the accuracy. The model received its feature matrix through Feast historical feature retrieval.

**Materialize Features to the Online Store**

In [ ]:
%cd /content/cse_skillgap_feast

!feast materialize \\
    2026-01-01T00:00:00 \\
    2026-01-01T00:10:00


**Retrieve Online Features**

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {
            "skill_id": "SKILL_025"
        }
    ]
).to_dict()


In [ ]:
print(online_features)


In [ ]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)


**Make an Online Prediction**

In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)


In [ ]:
X_online = online_df[
    feature_columns
]


In [ ]:
prediction = final_model.predict(
    X_online
)

print(
    "Predicted employability:",
    prediction[0]
)


**Retrieve Features for Multiple CSE Skill Records**

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"skill_id": "SKILL_010"},
        {"skill_id": "SKILL_020"},
        {"skill_id": "SKILL_030"},
        {"skill_id": "SKILL_040"}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)


In [ ]:
online_df["predicted_employability"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

display(
    online_df[
        [
            "skill_id",
            "predicted_employability"
        ]
    ]
)
